# Prueba F para Comparación de Dos Varianzas
Este notebook realiza una prueba de hipótesis para comparar las varianzas entre el dataset `df_limpio_2.csv` y el dataset gubernamental `conapesca_historico_2015_2025_LIMPIO.csv`.

## 1. Planteamiento de Hipótesis y Supuestos
**Hipótesis (Prueba Bilateral):**
* **Hipótesis Nula ($H_0$):** $\sigma_1^2 = \sigma_2^2$ (Las varianzas de ambas poblaciones son iguales).
* **Hipótesis Alternativa ($H_1$):** $\sigma_1^2 \neq \sigma_2^2$ (Las varianzas son diferentes).

**Nivel de significancia:** $\alpha = 0.05$ (5%)

**Revisión de Supuestos:**
1. **Aleatoriedad:** Las muestras se consideran aleatorias y representativas de las capturas.
2. **Independencia dentro de una muestra:** Conocer el valor de un registro no determina el valor de otro.
3. **Independencia entre dos muestras:** Los registros provienen de bases de datos y momentos diferentes, no hay emparejamiento.
4. **Normalidad:** Revisaremos gráficamente si las distribuciones son aproximadamente normales o si el teorema del límite central las ampara debido a la gran cantidad de datos.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

In [ ]:
# Cargar los datasets
try:
    df_nuestro = pd.read_csv('df_limpio_2.csv')
    df_conapesca = pd.read_csv('conapesca_historico_2015_2025_LIMPIO.csv', low_memory=False)
    
    # Extraer las muestras a comparar. Dado que CONAPESCA tiene pesos desembarcados, 
    # compararemos la varianza de la variable Peso.
    # Usaremos 'weight' de df_limpio_2 y 'PESO_VIVO_KG' (o la columna equivalente de peso) de CONAPESCA.
    muestra_1 = df_nuestro['weight'].dropna()
    muestra_2 = df_conapesca['PESO_VIVO_KG'].dropna() # Asegúrate que el nombre de columna coincida
    
    print(f'Muestra 1 (Nuestro Dataset) - Tamaño (n1): {len(muestra_1)}')
    print(f'Muestra 2 (Conapesca) - Tamaño (n2): {len(muestra_2)}')
except Exception as e:
    print(f'Error al cargar los datos: {e}\nAsegúrate de tener los archivos CSV en la misma carpeta.')

## 2. Cálculos Estadísticos (Prueba F)

In [ ]:
if 'muestra_1' in locals() and 'muestra_2' in locals():
    n1 = len(muestra_1)
    n2 = len(muestra_2)

    # Varianzas muestrales
    var1 = np.var(muestra_1, ddof=1)
    var2 = np.var(muestra_2, ddof=1)

    print(f'Varianza Muestra 1 ($S_1^2$): {var1:,.2f}')
    print(f'Varianza Muestra 2 ($S_2^2$): {var2:,.2f}')

    # Cálculo del Estadístico F: Varianza Mayor / Varianza Menor
    if var1 > var2:
        f_stat = var1 / var2
        dfn = n1 - 1
        dfd = n2 - 1
    else:
        f_stat = var2 / var1
        dfn = n2 - 1
        dfd = n1 - 1

    print(f'\nEstadístico F ($F_p$): {f_stat:,.4f}')
    print(f'Grados de libertad numerador (gl1): {dfn}')
    print(f'Grados de libertad denominador (gl2): {dfd}')

## 3. Valores Críticos y P-Value
Calculamos las regiones de rechazo basadas en una prueba bilateral (dos colas).

In [ ]:
if 'f_stat' in locals():
    alpha = 0.05

    # P-value para dos colas
    p_value = 2 * (1 - stats.f.cdf(f_stat, dfn, dfd))

    # Valores críticos
    f_crit_inf = stats.f.ppf(alpha / 2, dfn, dfd)
    f_crit_sup = stats.f.ppf(1 - alpha / 2, dfn, dfd)

    print(f'Nivel de Significancia (\u03B1): {alpha}')
    print(f'Valor P: {p_value:e}')
    print(f'Valor Crítico Inferior ($F_{{inf}}$): {f_crit_inf:,.4f}')
    print(f'Valor Crítico Superior ($F_{{sup}}$): {f_crit_sup:,.4f}')

## 4. Interpretación, Errores y Decisión

**Regla de Decisión:**
Rechazar $H_0$ si $F_p < F_{inf}$ o $F_p > F_{sup}$

**Tipos de Error:**
* **Error Tipo I (Falso Positivo):** Ocurre si rechazamos $H_0$ cuando en realidad las varianzas de ambas poblaciones sí son iguales. La probabilidad máxima de cometerlo es nuestro nivel de significancia $\alpha$ (5%).
* **Error Tipo II (Falso Negativo):** Ocurre si no rechazamos $H_0$ cuando en realidad las varianzas son diferentes.

In [ ]:
if 'f_stat' in locals():
    print("--- CONCLUSIÓN DE LA PRUEBA ---")
    if f_stat < f_crit_inf or f_stat > f_crit_sup:
        decision = 'RECHAZAR la Hipótesis Nula (H0).'
        interpretacion = 'Existe evidencia estadística significativa (con un 95% de confianza) para concluir que las varianzas poblacionales son diferentes.'
        error_riesgo = 'El error que podríamos estar cometiendo es el Error Tipo I (rechazar una igualdad que era cierta en la realidad).'
    else:
        decision = 'NO RECHAZAR la Hipótesis Nula (H0).'
        interpretacion = 'No hay suficiente evidencia para afirmar que las varianzas son diferentes. Asumimos que la variabilidad es estadísticamente similar.'
        error_riesgo = 'El error que podríamos estar cometiendo es el Error Tipo II (aceptar una igualdad cuando en realidad había diferencia).'

    print(f'Decisión: {decision}')
    print(f'Interpretación: {interpretacion}')
    print(f'Riesgo Actual: {error_riesgo}')

## 5. Gráficas y Visualización

In [ ]:
if 'muestra_1' in locals():
    # Filtrar outliers extremos solo para la visualización, para que las gráficas sean legibles
    m1_viz = muestra_1[muestra_1 < muestra_1.quantile(0.99)]
    m2_viz = muestra_2[muestra_2 < muestra_2.quantile(0.99)]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Histogramas (Revisión de Normalidad)
    sns.histplot(m1_viz, kde=True, ax=axes[0,0], color='blue', bins=30)
    axes[0,0].set_title('Distribución Muestra 1 (Nuestro Dataset)')

    sns.histplot(m2_viz, kde=True, ax=axes[0,1], color='green', bins=30)
    axes[0,1].set_title('Distribución Muestra 2 (Conapesca)')

    # Q-Q Plots (Revisión de Normalidad)
    stats.probplot(m1_viz.sample(min(1000, len(m1_viz)), random_state=42), dist='norm', plot=axes[1,0])
    axes[1,0].set_title('Q-Q Plot Muestra 1')

    stats.probplot(m2_viz.sample(min(1000, len(m2_viz)), random_state=42), dist='norm', plot=axes[1,1])
    axes[1,1].set_title('Q-Q Plot Muestra 2')

    plt.tight_layout()
    plt.show()

In [ ]:
if 'f_stat' in locals():
    plt.figure(figsize=(10, 6))
    
    # Crear la distribución F para el gráfico
    # Limitamos el eje X para que el gráfico sea visible, incluso si F es muy grande
    x_max = max(3, f_stat * 1.2, f_crit_sup * 1.5)
    x = np.linspace(0.01, x_max, 1000)
    y = stats.f.pdf(x, dfn, dfd)
    
    plt.plot(x, y, 'k-', linewidth=2, label=f'Distribución F (gl1={dfn}, gl2={dfd})')

    # Sombreado de regiones de rechazo
    x_inf = np.linspace(0.01, f_crit_inf, 100)
    plt.fill_between(x_inf, stats.f.pdf(x_inf, dfn, dfd), alpha=0.5, color='red', label=f'Región Rechazo Izq ({f_crit_inf:.4f})')

    # Si f_crit_sup es menor que el límite que pusimos en x_max
    if f_crit_sup < x_max:
        x_sup = np.linspace(f_crit_sup, x_max, 100)
        plt.fill_between(x_sup, stats.f.pdf(x_sup, dfn, dfd), alpha=0.5, color='red', label=f'Región Rechazo Der ({f_crit_sup:.4f})')

    # Línea del Estadístico Calculado
    plt.axvline(f_stat, color='blue', linestyle='--', linewidth=2.5, label=f'Estadístico $F_p$ = {f_stat:.4f}')

    plt.title('Prueba F Bilateral - Distribución y Regiones de Rechazo')
    plt.xlabel('Valor de F')
    plt.ylabel('Densidad')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()